# Complaint Copilot — Part 2: Extraction, Embeddings & RAG

Turns complaint narratives into structured fields with an LLM, embeds them for semantic
search, and answers natural-language questions grounded in the retrieved complaints (RAG).

**The LLM is optional.** If an `ANTHROPIC_API_KEY` is set, you get real LLM extraction and
generation. If not, every cell still runs using lightweight fallbacks, so you can see the
whole pipeline end-to-end without any key.

- Embeddings: `sentence-transformers/all-MiniLM-L6-v2` (downloads once, runs locally)
- Anthropic API docs: https://docs.claude.com/en/api/overview

In [ ]:
# Run once if needed:
# %pip install sentence-transformers numpy pandas pyarrow anthropic

In [ ]:
import os, json, re
import numpy as np
import pandas as pd

USE_LLM = bool(os.environ.get("ANTHROPIC_API_KEY"))   # auto-on when a key is present
EXTRACT_MODEL = "claude-haiku-4-5-20251001"   # cheap + fast for bulk extraction
ANSWER_MODEL  = "claude-haiku-4-5-20251001"   # swap to "claude-sonnet-4-6" for richer answers
EMBED_MODEL   = "sentence-transformers/all-MiniLM-L6-v2"
N_EXTRACT = 25                                 # how many to run extraction on for a quick run
DATA_PATH = os.path.join("data", "complaints.parquet")
print("USE_LLM =", USE_LLM)

USE_LLM = False


In [ ]:
def _sample_rows():
    # Synthetic fallback data so the notebook runs with no network. NOT real CFPB data.
    return [
        {"complaint_id":"S-0001","date_received":"2024-03-02","product":"Credit reporting","sub_product":"Credit reporting","issue":"Incorrect information on your report","sub_issue":"Account is not mine","company":"Acme Bureau","state":"WA","company_response":"Closed with explanation","timely":"Yes","complaint_what_happened":"I reported a fraudulent account three times but the bureau keeps saying it is verified and never explains how they verified it. This account is not mine and is hurting my score."},
        {"complaint_id":"S-0002","date_received":"2024-03-04","product":"Credit reporting","sub_product":"Credit reporting","issue":"Incorrect information on your report","sub_issue":"Old information reappears","company":"Acme Bureau","state":"CA","company_response":"Closed with explanation","timely":"Yes","complaint_what_happened":"A paid collection is still showing as open after 60 days even though I sent the confirmation letter that it was settled in full."},
        {"complaint_id":"S-0003","date_received":"2024-02-19","product":"Debt collection","sub_product":"Other debt","issue":"Attempts to collect debt not owed","sub_issue":"Debt is not yours","company":"Recovery Partners","state":"TX","company_response":"Closed with non-monetary relief","timely":"No","complaint_what_happened":"There was no response from the company for over a month after I disputed the amount owed. They keep adding fees that were never explained."},
        {"complaint_id":"S-0004","date_received":"2024-02-22","product":"Debt collection","sub_product":"Credit card debt","issue":"Communication tactics","sub_issue":"Frequent or repeated calls","company":"Recovery Partners","state":"FL","company_response":"Closed with explanation","timely":"Yes","complaint_what_happened":"They kept calling me about a debt I already settled. The file was never closed and I get calls multiple times a day."},
        {"complaint_id":"S-0005","date_received":"2024-03-10","product":"Mortgage","sub_product":"Conventional home mortgage","issue":"Trouble during payment process","sub_issue":"Escrow","company":"HomeServe Bank","state":"WA","company_response":"Closed with explanation","timely":"Yes","complaint_what_happened":"My mortgage servicer changed my escrow and doubled my payment without any notice. When I call, no one can explain the new amount."},
        {"complaint_id":"S-0006","date_received":"2024-03-12","product":"Mortgage","sub_product":"FHA mortgage","issue":"Applying for a mortgage","sub_issue":"Delays","company":"HomeServe Bank","state":"OR","company_response":"Closed with explanation","timely":"Yes","complaint_what_happened":"The loan officer lost my documents twice and the closing was delayed by weeks, which almost cost me the house."},
        {"complaint_id":"S-0007","date_received":"2024-03-15","product":"Money transfer","sub_product":"Mobile wallet","issue":"Fraud or scam","sub_issue":"Money sent to wrong person","company":"PayQuick","state":"NY","company_response":"Closed with explanation","timely":"Yes","complaint_what_happened":"I sent money to the wrong person through the app and support says nothing can be done to recover it. There is no way to reverse the transfer."},
        {"complaint_id":"S-0008","date_received":"2024-03-16","product":"Money transfer","sub_product":"Mobile wallet","issue":"Other transaction problem","sub_issue":"Transfer not received","company":"PayQuick","state":"GA","company_response":"In progress","timely":"Yes","complaint_what_happened":"The transfer was marked complete but the recipient never received it. It has been a week and the funds are just gone."},
        {"complaint_id":"S-0009","date_received":"2024-02-28","product":"Credit card","sub_product":"General purpose card","issue":"Problem with a purchase shown on statement","sub_issue":"Billing dispute","company":"Metro Card","state":"IL","company_response":"Closed with monetary relief","timely":"Yes","complaint_what_happened":"I was charged twice for the same purchase and the billing dispute was denied without any real review of the evidence I uploaded."},
        {"complaint_id":"S-0010","date_received":"2024-03-01","product":"Credit card","sub_product":"Store card","issue":"Fees or interest","sub_issue":"Unexpected interest","company":"Metro Card","state":"WA","company_response":"Closed with explanation","timely":"Yes","complaint_what_happened":"They charged me interest even though I paid the full statement balance before the due date. The interest keeps compounding."},
        {"complaint_id":"S-0011","date_received":"2024-03-20","product":"Student loan","sub_product":"Federal student loan servicing","issue":"Dealing with your lender or servicer","sub_issue":"Payment not applied","company":"EduServ","state":"PA","company_response":"Closed with explanation","timely":"Yes","complaint_what_happened":"My payments are not being applied to the principal and the servicer keeps putting me in forbearance that I never requested."},
        {"complaint_id":"S-0012","date_received":"2024-03-21","product":"Checking account","sub_product":"Checking account","issue":"Managing an account","sub_issue":"Funds not available","company":"First River Bank","state":"WA","company_response":"Closed with explanation","timely":"No","complaint_what_happened":"The bank placed a hold on my deposit for ten days with no explanation and I could not pay my rent because of it."},
        {"complaint_id":"S-0013","date_received":"2024-03-22","product":"Credit reporting","sub_product":"Credit reporting","issue":"Improper use of your report","sub_issue":"Hard inquiry","company":"Acme Bureau","state":"NV","company_response":"Closed with explanation","timely":"Yes","complaint_what_happened":"There are several hard inquiries on my report that I never authorized. I think someone is using my identity to open accounts."},
        {"complaint_id":"S-0014","date_received":"2024-03-25","product":"Money transfer","sub_product":"International transfer","issue":"Other transaction problem","sub_issue":"Excessive fees","company":"PayQuick","state":"CA","company_response":"Closed with explanation","timely":"Yes","complaint_what_happened":"The exchange rate and hidden fees took almost 15 percent of what I sent abroad and none of it was disclosed up front."},
    ]

def _sample_df():
    df = pd.DataFrame(_sample_rows()).rename(columns={"complaint_what_happened": "narrative"})
    return df

if os.path.exists(DATA_PATH):
    df = pd.read_parquet(DATA_PATH)
    print("Loaded", len(df), "complaints from", DATA_PATH)
else:
    df = _sample_df()
    print("No parquet found; using built-in sample of", len(df),
          "complaints. (Run notebook 01 to pull real data.)")
df = df.reset_index(drop=True)
df.head(2)

No parquet found; using built-in sample of 14 complaints. (Run notebook 01 to pull real data.)


,complaint_id,date_received,product,sub_product,issue,sub_issue,company,state,company_response,timely,narrative
0,S-0001,2024-03-02,Credit reporting,Credit reporting,Incorrect information on your report,Account is not mine,Acme Bureau,WA,Closed with explanation,Yes,I reported a fraudulent account three times bu...
1,S-0002,2024-03-04,Credit reporting,Credit reporting,Incorrect information on your report,Old information reappears,Acme Bureau,CA,Closed with explanation,Yes,A paid collection is still showing as open aft...


In [ ]:
def llm(prompt, model, max_tokens=500):
    # Returns model text, or None if USE_LLM is False or the call fails.
    if not USE_LLM:
        return None
    try:
        import anthropic
        client = anthropic.Anthropic()   # reads ANTHROPIC_API_KEY from the environment
        resp = client.messages.create(
            model=model,
            max_tokens=max_tokens,
            messages=[{"role": "user", "content": prompt}],
        )
        return resp.content[0].text
    except Exception as e:
        print("LLM call failed:", e)
        return None

## Step 1 — Structured extraction
Turn each free-text narrative into structured fields. Uses the LLM when available,
otherwise a transparent keyword heuristic.

In [ ]:
def heuristic_extract(text):
    t = str(text).lower()
    neg = any(w in t for w in ["never","refuse","fraud","wrong","still","no response","ignored","denied","hidden"])
    sentiment = "negative" if neg else "neutral"
    urgency = "high" if any(w in t for w in ["fraud","identity","stolen","urgent","immediately"]) else "normal"
    summary = str(text).strip().split(". ")[0][:140]
    return {"issue_summary": summary, "root_cause": "unspecified",
            "sentiment": sentiment, "urgency": urgency}

def llm_extract(text):
    prompt = (
        "Extract structured fields from this consumer complaint. "
        "Respond with ONLY a JSON object, no preamble, with keys: "
        "issue_summary (<=15 words), root_cause (<=10 words), "
        "sentiment (negative|neutral|positive), urgency (low|normal|high).\n\n"
        "Complaint:\n" + str(text)[:2000]
    )
    out = llm(prompt, EXTRACT_MODEL, max_tokens=200)
    if not out:
        return heuristic_extract(text)
    m = re.search(r"\{.*\}", out, re.S)
    try:
        return json.loads(m.group(0)) if m else heuristic_extract(text)
    except Exception:
        return heuristic_extract(text)

def extract(text):
    return llm_extract(text) if USE_LLM else heuristic_extract(text)

sub = df.head(N_EXTRACT).copy()
fields = sub["narrative"].apply(extract).apply(pd.Series)
extracted = pd.concat([sub[["complaint_id","product","narrative"]].reset_index(drop=True),
                       fields], axis=1)
extracted.head(5)

,complaint_id,product,narrative,issue_summary,root_cause,sentiment,urgency
0,S-0001,Credit reporting,I reported a fraudulent account three times bu...,I reported a fraudulent account three times bu...,unspecified,negative,high
1,S-0002,Credit reporting,A paid collection is still showing as open aft...,A paid collection is still showing as open aft...,unspecified,negative,normal
2,S-0003,Debt collection,There was no response from the company for ove...,There was no response from the company for ove...,unspecified,negative,normal
3,S-0004,Debt collection,They kept calling me about a debt I already se...,They kept calling me about a debt I already se...,unspecified,negative,normal
4,S-0005,Mortgage,My mortgage servicer changed my escrow and dou...,My mortgage servicer changed my escrow and dou...,unspecified,neutral,normal


## Step 2 — Embeddings + vector index
Embed every narrative once; retrieval is a cosine similarity over normalized vectors
(numpy here for zero extra dependencies; swap in FAISS or Chroma at scale).

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMBED_MODEL)
texts = df["narrative"].astype(str).tolist()
emb = embedder.encode(texts, normalize_embeddings=True, show_progress_bar=True)
emb = np.asarray(emb, dtype="float32")
print("Embedded", emb.shape[0], "complaints into", emb.shape[1], "dims")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedded 14 complaints into 384 dims


## Step 3 — RAG query engine
Retrieve the most similar complaints, then answer grounded only in those, with citations.

In [ ]:
def retrieve(query, k=4):
    q = embedder.encode([query], normalize_embeddings=True)[0].astype("float32")
    scores = emb @ q                      # cosine similarity (vectors are normalized)
    idx = np.argsort(-scores)[:k]
    hits = df.iloc[idx].copy()
    hits["score"] = scores[idx]
    return hits

retrieve("problems with credit report disputes", k=3)[["complaint_id","product","score","narrative"]]

,complaint_id,product,score,narrative
8,S-0009,Credit card,0.505049,I was charged twice for the same purchase and ...
0,S-0001,Credit reporting,0.419024,I reported a fraudulent account three times bu...
2,S-0003,Debt collection,0.406271,There was no response from the company for ove...


In [ ]:
def answer(query, k=4):
    hits = retrieve(query, k=k)
    context = "\n\n".join(
        "[" + str(r.complaint_id) + " | " + str(r.product) + "] " + str(r.narrative)[:500]
        for r in hits.itertuples()
    )
    if USE_LLM:
        prompt = (
            "You are an analyst answering questions about consumer complaints. "
            "Use ONLY the complaints below. Cite the complaint ids you used in brackets. "
            "If the complaints do not support an answer, say so.\n\n"
            "Question: " + query + "\n\nComplaints:\n" + context
        )
        out = llm(prompt, ANSWER_MODEL, max_tokens=400)
        if out:
            return out, hits
    summary = ("Based on " + str(len(hits)) + " retrieved complaints, the most relevant "
               "issues involve: " + "; ".join(hits["product"].astype(str).tolist()) + ".")
    return summary, hits

ans, src = answer("Which complaints involve money sent to the wrong person?")
print(ans)
print("\nSources:")
for r in src.itertuples():
    print("  ", r.complaint_id, "|", r.product, "| score =", round(float(r.score), 3))

Based on 4 retrieved complaints, the most relevant issues involve: Credit card; Debt collection; Money transfer; Money transfer.

Sources:
   S-0009 | Credit card | score = 0.455
   S-0003 | Debt collection | score = 0.446
   S-0007 | Money transfer | score = 0.395
   S-0008 | Money transfer | score = 0.324


In [ ]:
for q in ["credit report has wrong information",
          "company never responded to my dispute",
          "emerging problems with mobile payments"]:
    a, s = answer(q, k=3)
    print("Q:", q)
    print("A:", a)
    print("sources:", [r.complaint_id for r in s.itertuples()])
    print("-" * 60)

Q: credit report has wrong information
A: Based on 3 retrieved complaints, the most relevant issues involve: Credit reporting; Credit reporting; Credit card.
sources: ['S-0001', 'S-0013', 'S-0009']
------------------------------------------------------------
Q: company never responded to my dispute
A: Based on 3 retrieved complaints, the most relevant issues involve: Debt collection; Credit card; Debt collection.
sources: ['S-0003', 'S-0009', 'S-0004']
------------------------------------------------------------
Q: emerging problems with mobile payments
A: Based on 3 retrieved complaints, the most relevant issues involve: Money transfer; Student loan; Credit card.
sources: ['S-0007', 'S-0011', 'S-0009']
------------------------------------------------------------


## Step 4 — LLM-as-judge (lightweight eval)
A minimal faithfulness check: does the answer stay supported by the retrieved complaints?
Runs only when the LLM is enabled.

In [ ]:
def judge(query, ans_text, hits):
    if not USE_LLM:
        return "skipped (USE_LLM is False)"
    ctx = "\n".join("[" + str(r.complaint_id) + "] " + str(r.narrative)[:300] for r in hits.itertuples())
    prompt = ("Rate 1-5 how well the ANSWER is supported by the COMPLAINTS "
              "(5 = fully grounded, 1 = unsupported). Respond with only the number.\n\n"
              "QUESTION: " + query + "\n\nANSWER: " + ans_text + "\n\nCOMPLAINTS:\n" + ctx)
    out = llm(prompt, ANSWER_MODEL, max_tokens=5)
    return (out or "").strip()

q = "credit report shows an account that is not mine"
a, s = answer(q, k=3)
print("Faithfulness (1-5):", judge(q, a, s))

Faithfulness (1-5): skipped (USE_LLM is False)


## Where to go next
- Scale extraction across the full dataset and store fields in a warehouse (DuckDB / Postgres).
- Swap the numpy search for **FAISS** or **Chroma** as the corpus grows.
- Add unsupervised clustering (e.g. **BERTopic**) over the embeddings to surface *emerging* issue clusters.
- Wrap `answer()` in a **Streamlit** app to recreate the dashboard concept.

This is a quick-run scaffold — the real project scales each stage up.